# Symbol-spaced ZFE and MMSE-LE: channel, poles and noise

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Channel, energy and fixed equalizers

In [ ]:
from adsp.equalization.analysis import inverse_polynomial_residues

h = np.array([-0.5, 1.0 + 0.25j, -0.5j])
symbol_energy, snr_mfb, n_samples, seed = 1.0, 10.0, 50_000, 3502
energy = float(np.vdot(h, h).real)
noise_var = symbol_energy * energy / snr_mfb
zfe = ae.ZFE.fromChannel(h, n_taps=257, n_fft=65536)
mmse = ae.MMSELE.fromChannel(h, snr_mfb=snr_mfb, n_taps=257, n_fft=65536)
print("||h||²:", energy, "N0/2:", noise_var)
print("Q taps:", zfe.q)
for label, model, reg in (("ZFE", zfe, 0.0), ("MMSE-LE", mmse, 1 / snr_mfb)):
  poly, poles, residues, center = inverse_polynomial_residues(h, model.q, reg)
  print(label, "polynomial:", poly)
  print("D poles:", poles, "residues:", residues, "center:", center)
  assert np.isclose(center, model.idealCenterTap(), rtol=1e-10, atol=1e-11)

## 3. Data and fixed-model evaluation

In [ ]:
batch = data.matched_filter(n_samples, rng=np.random.default_rng(seed), equalizer=zfe, h=h, symbol_energy=symbol_energy,
                            noise_var=noise_var)
results = {
  label: training.evaluate(model, batch, name=label, guard=160)
  for label, model in (("ZFE", zfe), ("MMSE-LE", mmse))
}
components = mmse.errorComponents(symbol_energy, noise_var)
ideal_mse = {"ZFE": zfe.outputNoiseVariance(noise_var), "MMSE-LE": mmse.mse(symbol_energy, noise_var)}
assert np.isclose(ideal_mse["MMSE-LE"], components["sum"])
assert np.isclose(ideal_mse["MMSE-LE"], mmse.mseFromCenterTap(noise_var))
for label, result in results.items():
  measured = np.mean(result.mse[result.valid])
  print(f"{label}: measured MSE={measured:.8g}, ideal={ideal_mse[label]:.8g}")
  assert np.isclose(measured, ideal_mse[label], rtol=0.075)
print("ZFE SNR:", zfe.snr(symbol_energy, noise_var))
print("MMSE unbiased SNR:", mmse.unbiasedSnr(symbol_energy, noise_var))
print("MMSE components:", components)

## 4. Unified dark figures

In [ ]:
frequency = np.linspace(-0.5, 0.5, 2001)
h_response = ae.BaseEqualizer.responseFromTaps(h, np.array([-1, 0, 1]), frequency)
fig, grid = plot.axes(rows=2, cols=2)
plot.lines({"Channel": (frequency, h_response)}, ax=grid[0, 0], scale="amplitude_db", title="Channel magnitude",
           xlabel="Cycles/symbol", ylabel="Magnitude (dB)")
spectra, cascades = {}, {}
for label, model in (("ZFE", zfe), ("MMSE-LE", mmse)):
  omega, response = model.frequencyResponse(centered=True)
  spectra[label] = (omega / np.pi, response)
  cascades[label] = (omega / np.pi, np.abs(model.h_norm * response * np.fft.fftshift(model._q_bins)))
plot.lines(spectra, ax=grid[0, 1], scale="amplitude_db", xlabel="Ω / π", ylabel="Magnitude (dB)",
           title="Ideal equalizer magnitude")
plot.lines(cascades, ax=grid[1, 0], xlabel="Ω / π", title="Equalized matched-filter channel")
keep = np.abs(zfe.w_index) <= 12
plot.lines({
  "ZFE direct convention": np.abs(zfe.w[keep]),
  "MMSE-LE": np.abs(mmse.w[keep])
}, x=zfe.w_index[keep], ax=grid[1, 1], xlabel="Two-sided tap lag", title="FIR truncation magnitudes")
plot.show(fig)
# Preserve the original alternating display convention as an explicit comparison.
plot.stem({
  "Direct real taps": zfe.w[keep].real,
  "Alternating display": (zfe.w[keep] * (-1.0)**zfe.w_index[keep]).real
}, x=zfe.w_index[keep], title="Direct inverse-DTFT and alternating tap display")
plot.show()
training.summary(results, tail=5000)